# Onboarding emails and cancellations: one row per customer

**Kaplan-Meier Method (Database Format)** · a worked example from the StatNex teaching guide

**The story.** A streaming app gave 42 people a free trial. Half of them received a series of helpful onboarding emails, the other half did not. Each person was followed for up to 26 weeks. Here the data are laid out the way a company database stores them: one row per customer, with a group column.

**The question.** Do onboarding emails delay cancellations?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `07_survival/kaplan_meier_database.csv` (42 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Group,Week,Status
With emails,3.0,1.0
With emails,5.0,1.0
With emails,7.0,1.0
With emails,8.0,1.0
With emails,11.0,1.0
With emails,14.0,1.0
With emails,18.0,1.0
With emails,22.0,1.0
With emails,6.0,0.0
With emails,9.0,0.0
With emails,12.0,0.0
With emails,15.0,0.0
With emails,16.0,0.0
With emails,20.0,0.0
With emails,24.0,0.0
With emails,26.0,0.0
With emails,26.0,0.0
With emails,26.0,0.0
With emails,26.0,0.0
With emails,26.0,0.0
With emails,26.0,0.0
Without emails,1.0,1.0
Without emails,1.0,1.0
Without emails,2.0,1.0
Without emails,3.0,1.0
Without emails,3.0,1.0
Without emails,4.0,1.0
Without emails,5.0,1.0
Without emails,5.0,1.0
Without emails,6.0,1.0
Without emails,7.0,1.0
Without emails,8.0,1.0
Without emails,8.0,1.0
Without emails,9.0,1.0
Without emails,10.0,1.0
Without emails,12.0,1.0
Without emails,13.0,1.0
Without emails,15.0,1.0
Without emails,17.0,1.0
Without emails,20.0,1.0
Without emails,26.0,0.0
Without emails,26.0,0.0

These cells save `logrank_tests.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile logrank_tests.py
"""The four tests that compare survival curves, written out.

At every event time each test compares the events a group had with the events it would
have had if every group shared one survival curve. They differ in how much each event
time counts, and Peto-Peto also in its variance:

* Cochran-Mantel-Haenszel (the log-rank test): every event time counts the same;
* Gehan-Breslow (generalised Wilcoxon): weighted by the number at risk, so early times
  count more;
* Peto-Prentice: weighted by S~(t) = prod(1 - d / (n + 1)), a survival estimate;
* Peto-Peto (Peto and Peto 1972): each case scored from the pooled Kaplan-Meier curve,
  with the permutation variance of the scores (the variance of the score sums when the
  group labels are shuffled), as R's coin::logrank_test(type = "Peto-Peto").

The analysis takes the first two from statsmodels' survdiff(), which gives the same
numbers; they stay here as a check. statsmodels has neither Peto test, so the last two
are needed. Stata's -sts test, peto- is the Peto-Prentice test here.

With k groups the statistic is a quadratic form on k - 1 degrees of freedom; with two
groups it is U^2 / V. Event times where fewer than two are at risk carry no information
and are left out of the weighted tests.
"""

import numpy as np
from scipy import stats


def counts_at(times, events, grid):
    """Number at risk (time >= t) and number of events at each time t of ``grid``."""
    at_risk = len(times) - np.searchsorted(np.sort(times), grid, side="left")
    event_times = np.sort(times[events == 1])
    deaths = (np.searchsorted(event_times, grid, side="right")
              - np.searchsorted(event_times, grid, side="left"))
    return at_risk.astype(float), deaths.astype(float)


def modified_survival(at_risk, deaths):
    """S~(t) = prod(1 - d / (n + 1)) over the event times up to and including t."""
    return np.cumprod(1 - deaths / (at_risk + 1))


def quadratic_form(u, v):
    """(U' V^-1 U, df) on the first k - 1 groups: the k sums add to zero, so V is singular
    and one group is dropped. df is the rank of what is left."""
    u, v = u[:-1], v[:-1, :-1]
    df = np.linalg.matrix_rank(v)
    statistic = float(u @ np.linalg.pinv(v) @ u)
    return statistic, df, stats.chi2.sf(statistic, df)


def weighted_logrank(times, events, codes, k, weight):
    """(chi-square, df, P); ``weight`` is "logrank", "gehan" or "peto_prentice"."""
    grid = np.unique(times[events == 1])
    n, d = counts_at(times, events, grid)
    w = {"logrank": np.ones(grid.size), "gehan": n,
         "peto_prentice": modified_survival(n, d)}[weight]
    usable = n >= 2
    n, d, w = n[usable], d[usable], w[usable]
    shares = np.empty((k, n.size))  # each group's share of the risk set
    u = np.empty(k)  # observed minus expected events, weighted
    for j in range(k):
        n_j, d_j = counts_at(times[codes == j], events[codes == j], grid)
        shares[j] = n_j[usable] / n
        u[j] = np.sum(w * (d_j[usable] - shares[j] * d))
    spread = w**2 * d * (n - d) / (n - 1)
    v = np.diag((spread * shares).sum(axis=1)) - (spread * shares) @ shares.T
    return quadratic_form(u, v)


def peto_peto(times, events, codes, k):
    """(chi-square, df, P) for Peto and Peto's (1972) test, with the permutation variance.

    S is the pooled Kaplan-Meier curve. An event at t scores S(t) + S(t-) - 1 and a case
    censored at t scores S(t) - 1, S(t) counting the events at t. The scores add to zero, so
    each group's sum is compared with its variance when the group labels are shuffled. R's
    coin::logrank_test(type = "Peto-Peto") gives the same chi-square."""
    grid = np.unique(times[events == 1])
    n, d = counts_at(times, events, grid)
    survival = np.cumprod(1 - d / n)
    before = np.concatenate([[1.0], survival[:-1]])
    last = np.searchsorted(grid, times, side="right") - 1  # last event time at or before
    at_time = np.where(last >= 0, survival[np.maximum(last, 0)], 1.0)
    scores = np.where(events == 1, at_time + before[np.maximum(last, 0)] - 1, at_time - 1)
    scores = scores - scores.mean()  # zero already, up to rounding
    total = len(times)
    sizes = np.bincount(codes, minlength=k).astype(float)
    u = np.bincount(codes, weights=scores, minlength=k)
    v = np.sum(scores**2) / (total * (total - 1)) * (total * np.diag(sizes) - np.outer(sizes, sizes))
    return quadratic_form(u, v)

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.duration.survfunc import SurvfuncRight, survdiff
from logrank_tests import peto_peto, weighted_logrank

pd.set_option("display.width", 140, "display.max_columns", None)  # show every column

## The data

In [ ]:
# One row per case: the time, the status, and the group. A row with any of the three
# blank is dropped. Set group = None for one curve over everybody.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
time, status, group = "Week", "Status", "Group"
event_value = "1"  # the status that records the event (death, failure...)
confidence = 95 / 100
z = stats.norm.ppf(0.5 + confidence / 2)

frame = data[[c for c in (time, status, group) if c]].dropna()
times = frame[time].to_numpy(float)


def is_event(column, value):
    """Numbers compare as numbers (1 equals 1.0); anything else compares as text."""
    numbers = pd.to_numeric(column, errors="coerce")
    target = pd.to_numeric(pd.Series([value]), errors="coerce")[0]
    if numbers.notna().all() and pd.notna(target):
        return (numbers == float(target)).to_numpy(int)
    return (column.astype(str).str.strip() == str(value).strip()).to_numpy(int)


events = is_event(frame[status], event_value)
labels = sorted(frame[group].unique()) if group else ["Overall"]
codes = (np.array([labels.index(g) for g in frame[group]]) if group
         else np.zeros(len(frame), dtype=int))

## Summary of cases

In [ ]:
summary = pd.DataFrame({"events": [events[codes == j].sum() for j in range(len(labels))],
                        "censored": [(events[codes == j] == 0).sum() for j in range(len(labels))],
                        "total": [(codes == j).sum() for j in range(len(labels))]}, index=labels)
print(summary)

## Kaplan-Meier estimate

In [ ]:
# The share still without the event, S(t), from statsmodels' SurvfuncRight: at each
# event time it multiplies the curve by (1 - d/n), d the events then and n the cases
# still at risk. Its standard error is Greenwood's. A censored case counts in n until
# its time and then leaves without moving the curve; that is how the method uses the
# cases whose event has not happened yet, rather than dropping them. The table has one
# row per distinct time, with a row for each time at which cases were only censored.


def kaplan_meier(t, e):
    table = (pd.DataFrame({"time": t, "event": e}).groupby("time")["event"]
             .agg(events="sum", cases="size"))
    table["censored"] = table["cases"] - table["events"]
    table["at risk"] = table["cases"][::-1].cumsum()[::-1]  # everyone still under observation
    fit = SurvfuncRight(t, e)
    table["survival"], table["SE"] = 1.0, 0.0  # before the first event the curve is at 1
    if len(fit.surv_times):
        # Each row takes the curve at the last event time at or before it. Where S has
        # reached 0, Greenwood's SE is 0/0 (NaN); it is reported as 0, the point itself.
        step = np.searchsorted(fit.surv_times, table.index.to_numpy(), side="right") - 1
        at = np.maximum(step, 0)
        table["survival"] = np.where(step >= 0, fit.surv_prob[at], 1.0)
        table["SE"] = np.where(step >= 0, np.nan_to_num(fit.surv_prob_se[at]), 0.0)
    return table.drop(columns="cases").reset_index(), fit


fitted = {label: kaplan_meier(times[codes == j], events[codes == j])
          for j, label in enumerate(labels)}
curves = {label: table for label, (table, _) in fitted.items()}

# The pointwise confidence band of each curve, printed with the survival table below;
# the median's interval is read off it.
# The interval S +- z SE, cut off at 0 and 1.
scale = "linear"  # the same scale, by its name in statsmodels
for table in curves.values():
    s, se = table["survival"], table["SE"]
    table["lower"] = np.maximum(0.0, s - z * se)
    table["upper"] = np.minimum(1.0, s + z * se)

## Mean and median survival time

In [ ]:
# The mean is the area under the curve up to the group's largest time (a restricted
# mean), with its standard error; when the largest time is an event that empties the
# risk set, the variance is scaled by D / (D - 1), D the number of events. The median is
# the first time at which S(t) is 0.5 or less. statsmodels has no restricted mean, so
# the area is added up here, step by step; R's survfit gives the same mean and SE.
# The median's standard error is Collett's (2003, section 2.4): the curve's Greenwood SE
# at the median divided by the density there, estimated as the drop of the curve
# between u, the last event time with S >= 0.55, and l, the first with S <= 0.45, over
# the time between them. It is left blank when the curve never falls to 0.45.
# The median's interval is Brookmeyer and Crowley's, read off the band above: the event
# times at which the band still holds 0.5. statsmodels' quantile_ci() finds them on the
# same scale as the band. The upper limit is blank (statsmodels: inf) where the band
# never falls below 0.5: the data cannot rule out a median beyond the follow-up.
# The median itself is kept by hand: SAS and statsmodels' quantile() take the first time
# S(t) is *below* 0.5, and R the middle of a flat stretch at exactly 0.5.


def median_se(table):
    events = table[table["events"] > 0]
    at_median = events[events["survival"] <= 0.5]
    upper = events[events["survival"] >= 0.55]
    lower = events[events["survival"] <= 0.45]
    if at_median.empty or upper.empty or lower.empty:
        return np.nan
    u, low = upper.iloc[-1], lower.iloc[0]
    density = (u["survival"] - low["survival"]) / (low["time"] - u["time"])
    return at_median["SE"].iloc[0] / density if density > 0 else np.nan


def median_interval(fit):
    if not len(fit.surv_times):
        return np.nan, np.nan
    with np.errstate(divide="ignore", invalid="ignore"):  # S = 0 has no transformed value
        low, high = fit.quantile_ci(0.5, alpha=1 - confidence, method=scale)
    return low, (np.nan if np.isinf(high) else high)


rows = []
for label, (table, fit) in fitted.items():
    t, s = table["time"].to_numpy(), table["survival"].to_numpy()
    n, d = table["at risk"].to_numpy(float), table["events"].to_numpy(float)
    before = np.concatenate([[1.0], s[:-1]])  # the curve's height over each step
    area = np.cumsum(before * np.diff(t, prepend=0.0))
    mean = area[-1]
    usable = (d > 0) & (n > d)
    variance = np.sum((mean - area[usable]) ** 2 * d[usable] / (n[usable] * (n[usable] - d[usable])))
    if d[-1] and n[-1] == d[-1] and d.sum() > 1:
        variance *= d.sum() / (d.sum() - 1)
    se = np.sqrt(variance)
    median = t[s <= 0.5][0] if (s <= 0.5).any() else np.nan
    median_error = median_se(table)
    rows.append([mean, se, mean - z * se, mean + z * se,
                 median, median_error, *median_interval(fit)])
survival_stats = pd.DataFrame(rows, index=labels, columns=[
    "mean", "SE", "lower", "upper", "median", "median SE", "median lower", "median upper"])
print()
print(survival_stats)

if len(labels) >= 2:
    # === Tests for a difference between the curves ===
    # Four tests of H0: every group has the same survival curve, on k - 1 df. At each
    # event time each compares a group's events with the events it would have had if
    # every group shared one curve; they differ in how much each time counts. The
    # log-rank test counts every time alike and is the most powerful when one group's
    # hazard is a constant multiple of the other's; the generalised Wilcoxon tests count
    # early times more. Choose one before looking. A large chi-square (small P) says the
    # curves differ somewhere; it does not say by how much.
    # statsmodels' survdiff() gives the log-rank and Gehan-Breslow ("gb") tests. It has
    # neither Peto test, so both are written out in logrank_tests.py; Peto-Peto there is
    # R's coin::logrank_test(type = "Peto-Peto").
    k = len(labels)

    def from_survdiff(weight_type=None):
        chi_square, p = survdiff(times, events, codes, weight_type=weight_type)
        return chi_square, k - 1, p

    tests = pd.DataFrame(
        [from_survdiff(),
         from_survdiff("gb"),
         peto_peto(times, events, codes, k),
         weighted_logrank(times, events, codes, k, "peto_prentice")],
        index=["Log-rank: Cochran-Mantel-Haenszel", "Generalised Wilcoxon: Gehan-Breslow",
               "Generalised Wilcoxon: Peto-Peto", "Generalised Wilcoxon: Peto-Prentice"],
        columns=["chi-square", "df", "P"])
    print()
    print(tests)

## Survival table

In [ ]:
survival_table = pd.concat(curves, names=["group", "row"])
print()
print(survival_table)

## What the result says

Here each of the 42 customers is one row, with a group, a week and a status. Status 1 means the customer cancelled. Status 0 means censored: still subscribed when we stopped watching. The Kaplan-Meier method uses both kinds of rows.

The result is the same as with one column per group. The mean survival time, the average time subscribed within the 26 weeks, is 19.7 weeks with emails and 9.6 weeks without. The log-rank test compares the two survival curves: chi-square = 12.94, and its P value, the chance of a gap at least this large if the groups were alike, is far below 0.01.

Email users stay about twice as long. Whether the emails are the cause depends on how people were put into the groups.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.